In [41]:
import numpy as np

h1 = 1
m = 0.5

def coupling(matrix_1, matrix_2):
    s11_1, s12_1 = matrix_1[0, 0], matrix_1[0, 1]
    s21_1, s22_1 = matrix_1[1, 0], matrix_1[1, 1]
    s11_2, s12_2 = matrix_2[0, 0], matrix_2[0, 1]
    s21_2, s22_2 = matrix_2[1, 0], matrix_2[1, 1]
    
    denom = 1 - s11_2 * s22_1
    s11 = s11_1 + s12_1 * s21_1 * s11_2 / denom
    s12 = s12_1 * s12_2 / denom
    s21 = s21_2 * s21_1 / denom
    s22 = s22_2 + s21_2 * s12_2 * s22_1 / denom
    
    return np.array([[s11, s12], [s21, s22]], dtype=np.complex128)

def multiple_coupling(matrix_collect):
    result = matrix_collect[0]
    for matrix in matrix_collect[1:]:
        result = coupling(result, matrix)
    return result

def coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a):
    constant_1 = 2 * m * (y1 + 1j * y2) / (h1**2)
    constant_2 = 2 * m * (y1 - 1j * y2) / (h1**2)
    
    di = np.array([constant_1 if i % 2 == 0 or i >= 2 * N and i < 2 * N + n3 else constant_2 
                   for i in range(2 * N + n3 + n4)], dtype=complex)
    
    x_0 = 0
    x_1 = a * (2 * N + n3 + n4) - a + x_0
    location = np.linspace(x_0, x_1, 2 * N + n3 + n4)
 #   print(location)
    k = np.sqrt(E * 2 * m) / h1
    exp_2jk = np.exp(2j * k * location)
    exp_minus_2jk = np.exp(-2j * k * location)
    
    matrix_collect = np.zeros((2 * N + n3 + n4, 2, 2), dtype=complex)
    for i in range(2 * N + n3 + n4):
        s11 = di[i] * exp_2jk[i]
        s12 = 2j * k
        s21 = 2j * k
        s22 = di[i] * exp_minus_2jk[i]
        
        matrix_collect[i] = np.array([[s11, s12], [s21, s22]]) / (-di[i] + 2j * k)
  #  print(matrix_collect)
    return multiple_coupling(matrix_collect)

def Transmission_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a):
    matrix = coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a)
    return float(np.abs(matrix[1, 0])**2)

def Reflection_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a):
    matrix = coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a)
    return float(np.abs(matrix[0, 0])**2)

In [42]:
import numpy as np     #AI成品
from sympy import symbols, nsolve, cos, sin, sqrt, pi
import matplotlib.pyplot as plt
import os

def calculate_energy_levels(N, y1, y2, a, E_min, E_max, tol=1e-6):
    """
    计算给定参数下的能量级别(E值)
    参数:
        N: 系统参数
        y1, y2: 势能参数
        a: 系统参数
        E_min, E_max: 能量范围
        tol: 容差
    返回:
        排序后的唯一E值列表
    """
    N_0 = N if N % 2 == 0 else N - 1
    n_values = np.arange(int(N_0/2 + 1))
    E_sym = symbols('E')
    all_solutions = []

    for n in n_values:
        equation = cos(2 * a * sqrt(E_sym)) + y1 * sin(2 * a * sqrt(E_sym)) / sqrt(E_sym) \
                   - (y1**2 + y2**2) * (cos(2 * a * sqrt(E_sym)) - 1) / (4 * E_sym) \
                   - cos(2 * pi * n / N)
        
        solutions = []
        for E_guess in np.linspace(E_min + 0.1, E_max, 100):
            try:
                sol = nsolve(equation, E_guess, tol=1e-6)
                if E_min <= sol <= E_max:
                    is_new = True
                    for existing in solutions:
                        if abs(existing - sol) < tol:
                            is_new = False
                            break
                    if is_new:
                        solutions.append(float(sol))
            except:
                continue
        
        solutions.sort()
        all_solutions.extend(solutions)

    return sorted(list(set(round(sol/tol)*tol for sol in all_solutions)))

def plot_transmission_reflection(N, y11, y22, a, zhi_1, E_ranges, n3=0, n4=0, m=0.5, h1=1):
    """
    主绘图函数
    参数:
        N, y11, y22, a, zhi_1: 系统参数
        E_ranges: 能量范围列表
        n3, n4, m, h1: 默认参数
    """
    # 创建输出目录
    folder_name = f"T&R实数与虚数_N_E={E_ranges}"
    output_dir = os.path.join(r"C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2", folder_name)
    os.makedirs(output_dir, exist_ok=True)

    # 遍历每个能量范围
    for range_idx, (E_min, E_max) in enumerate(E_ranges):
        # 动态子图布局
        num_rows = len(y11)
        num_cols = len(y22)
        fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 4 * num_rows))
        fig.suptitle(f'Transmission and Reflection Coefficients vs Energy ({E_min} to {E_max})', fontsize=16)

        # 处理单行/单列情况
        if num_rows == 1 and num_cols == 1:
            axes = np.array([[axes]])
        elif num_rows == 1:
            axes = axes.reshape(1, -1)
        elif num_cols == 1:
            axes = axes.reshape(-1, 1)

        # 遍历 y11 和 y22 的组合
        for i, y1 in enumerate(y11):
            for j, y2 in enumerate(y22):
                # 计算当前(y1,y2)对应的能量级别
                energy_levels = calculate_energy_levels(N, y1, y2, a, E_min, E_max)
                
                # 计算传输和反射系数
                E_values = np.linspace(E_min, E_max, 1000)
                T_values = [Transmission_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a) for E in E_values]
                R_values = [Reflection_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a) for E in E_values]
                Sum_values = [T + R for T, R in zip(T_values, R_values)]

                ax = axes[i, j]
                if 'T' in zhi_1:
                    ax.plot(E_values, T_values, label='Transmission (T)', color='blue')
                if 'R' in zhi_1:
                    ax.plot(E_values, R_values, label='Reflection (R)', color='red')
                if 'T+R' in zhi_1:
                    ax.plot(E_values, Sum_values, label='T + R', color='gray', linestyle='--', linewidth=0.8)

                # 添加能量级别线
                for x in energy_levels:
                    ax.axvline(x, color='g', linestyle='-', linewidth=0.5, alpha=0.3)

                ax.set_xlabel('Energy (E)')
                ax.set_ylabel('Coefficient Value')
                ax.set_title(f'y1={y1}, y2={y2}')
                ax.legend()
                ax.set_xlim(E_min, E_max)
                ax.set_ylim(0, 10)

        plt.tight_layout()

        # 保存图片
        y11_str = '_'.join(map(str, y11))
        y22_str = '_'.join(map(str, y22))
        zhi1_str = '_'.join(zhi_1)
        output_filename = os.path.join(
            output_dir,
            f'{range_idx + 1}_T&R({E_min},{E_max})_N={N}_n3={n3}_n4={n4}_y11={y11_str}_y22={y22_str}_a={a}_zhi1={zhi1_str}.png'
        )
        plt.savefig(output_filename, dpi=300)
        plt.close()
        print(f"已保存: {output_filename}")


In [43]:
# 使用示例
if __name__ == "__main__":
    # 系统参数
    y11 = [2]  
    y22 = [2]  
    a = 0.5
    zhi_1 = ['T'] #,'R', 'T+R']  # 选择绘制的曲线
    E_ranges = [(0, 50)]  # 可以分段计算
    for N in np.linspace(1,50,50).astype(int):
        plot_transmission_reflection(N, y11, y22, a, zhi_1, E_ranges)

C:\Users\taoji\AppData\Local\Temp\ipykernel_18572\2617171104.py:13: RuntimeWarning: invalid value encountered in scalar divide
  s11 = s11_1 + s12_1 * s21_1 * s11_2 / denom
C:\Users\taoji\AppData\Local\Temp\ipykernel_18572\2617171104.py:14: RuntimeWarning: invalid value encountered in scalar divide
  s12 = s12_1 * s12_2 / denom
C:\Users\taoji\AppData\Local\Temp\ipykernel_18572\2617171104.py:15: RuntimeWarning: invalid value encountered in scalar divide
  s21 = s21_2 * s21_1 / denom
C:\Users\taoji\AppData\Local\Temp\ipykernel_18572\2617171104.py:16: RuntimeWarning: invalid value encountered in scalar divide
  s22 = s22_2 + s21_2 * s12_2 * s22_1 / denom


已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R实数与虚数_N_E=[(0, 50)]\1_T&R(0,50)_N=1_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R实数与虚数_N_E=[(0, 50)]\1_T&R(0,50)_N=2_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R实数与虚数_N_E=[(0, 50)]\1_T&R(0,50)_N=3_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R实数与虚数_N_E=[(0, 50)]\1_T&R(0,50)_N=4_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R实数与虚数_N_E=[(0, 50)]\1_T&R(0,50)_N=5_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R实数与虚数_N_E=[(0, 50)]\1_T&R(0,50)_N=6_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R实数与虚数_N_E=[(0, 50)]\1_T&R(0,50)_N=7_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R实数与虚数_N_E=[(0, 50)]\1_T&R(0,50)_N=8_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
已保存: C:\

In [ ]:
import os                     # 最终AI成品            用于创建GIF 
from PIL import Image, ImageDraw, ImageFont
import imageio  # 用于创建GIF

# 设置路径和参数
input_dir = f"T&R实数与虚数_N_E={E_ranges}"
output_dir = r"C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2"
gif_name = "T&R_N1-50_Animation.gif"

# 获取所有图片文件并按N值排序
image_files = [f for f in os.listdir(input_dir) if f.startswith("1_T&R") and f.endswith(".png")]
image_files.sort(key=lambda x: int(x.split("N=")[1].split("_")[0]))

# 准备字体（使用默认字体或指定字体）
try:
    font = ImageFont.truetype("arial.ttf", 30)
except:
    font = ImageFont.load_default()

# 处理每张图片并创建帧列表
frames = []
for i, img_file in enumerate(image_files, start=1):
    img_path = os.path.join(input_dir, img_file)
    
    # 打开图片并添加文本
    with Image.open(img_path) as img:
        # 转换为RGB（如果是RGBA）
        if img.mode in ('RGBA', 'LA'):
            background = Image.new('RGB', img.size, (255, 255, 255))
            background.paste(img, mask=img.split()[-1])
            img = background
        
        draw = ImageDraw.Draw(img)
        
        # 添加N值文本（白色文字，黑色背景框）
        N_value = img_file.split("N=")[1].split("_")[0]
        text = f"N = {N_value}"
        
        # 计算文本位置（右上角） - 使用textbbox替代textsize
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        margin = 10
        x = img.width - text_width - margin
        y = margin
        
        # 绘制背景框
        draw.rectangle([x - margin, y - margin, 
                       x + text_width + margin, 
                       y + text_height + margin], fill="black")
        
        # 绘制文本
        draw.text((x, y), text, fill="white", font=font)
        
        # 添加到帧列表
        frames.append(img)

# 保存为GIF（调整duration控制播放速度）
gif_path = os.path.join(output_dir, gif_name)
imageio.mimsave(gif_path, frames, format='GIF', duration=1)  # 每帧0.5秒

print(f"动态图已保存至: {gif_path}")

In [46]:
import numpy as np

h1 = 1
m = 0.5

def coupling(matrix_1, matrix_2):
    s11_1, s12_1 = matrix_1[0, 0], matrix_1[0, 1]
    s21_1, s22_1 = matrix_1[1, 0], matrix_1[1, 1]
    s11_2, s12_2 = matrix_2[0, 0], matrix_2[0, 1]
    s21_2, s22_2 = matrix_2[1, 0], matrix_2[1, 1]
    
    denom = 1 - s11_2 * s22_1
    s11 = s11_1 + s12_1 * s21_1 * s11_2 / denom
    s12 = s12_1 * s12_2 / denom
    s21 = s21_2 * s21_1 / denom
    s22 = s22_2 + s21_2 * s12_2 * s22_1 / denom
    
    return np.array([[s11, s12], [s21, s22]], dtype=np.complex128)

def multiple_coupling(matrix_collect):
    result = matrix_collect[0]
    for matrix in matrix_collect[1:]:
        result = coupling(result, matrix)
    return result

def coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a):
    constant_1 = 2 * m * (y1 + 1j * y2) / (h1**2)
    constant_2 = 2 * m * (y1 - 1j * y2) / (h1**2)
    
    di = np.array([constant_1 if i % 2 == 0 or i >= 2 * N and i < 2 * N + n3 else constant_2 
                   for i in range(2 * N + n3 + n4)], dtype=complex)
    
    x_0 = 0
    x_1 = a * (2 * N + n3 + n4) - a + x_0
    location = np.linspace(x_0, x_1, 2 * N + n3 + n4)
 #   print(location)
    k = np.sqrt(E * 2 * m) / h1
    exp_2jk = np.exp(2j * k * location)
    exp_minus_2jk = np.exp(-2j * k * location)
    
    matrix_collect = np.zeros((2 * N + n3 + n4, 2, 2), dtype=complex)
    for i in range(2 * N + n3 + n4):
        s11 = di[i] * exp_2jk[i]
        s12 = 2j * k
        s21 = 2j * k
        s22 = di[i] * exp_minus_2jk[i]
        
        matrix_collect[i] = np.array([[s11, s12], [s21, s22]]) / (-di[i] + 2j * k)
  #  print(matrix_collect)
    return multiple_coupling(matrix_collect)

def Transmission_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a):
    matrix = coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a)
    return float(np.abs(matrix[1, 0])**2)

def Reflection_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a):
    matrix = coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a)
    return float(np.abs(matrix[0, 0])**2)

In [47]:
def calculate_energy_levels(N, y1, y2, a, E_min, E_max, tol=1e-6):
    """
    计算给定参数下的能量级别(E值)
    参数:
        N: 系统参数
        y1, y2: 势能参数
        a: 系统参数
        E_min, E_max: 能量范围
        tol: 容差
    返回:
        排序后的唯一E值列表
    """
    N_0 = N if N % 2 == 0 else N - 1
    n_values = np.arange(int(N_0/2 + 1))
    E_sym = symbols('E')
    all_solutions = []

    for n in n_values:
        equation = cos(2 * a * sqrt(E_sym)) + y1 * sin(2 * a * sqrt(E_sym)) / sqrt(E_sym) \
                   - (y1**2 + y2**2) * (cos(2 * a * sqrt(E_sym)) - 1) / (4 * E_sym) \
                   - cos(2 * pi * n / N)
        
        solutions = []
        for E_guess in np.linspace(E_min + 0.1, E_max, 100):
            try:
                sol = nsolve(equation, E_guess, tol=1e-6)
                if E_min <= sol <= E_max:
                    is_new = True
                    for existing in solutions:
                        if abs(existing - sol) < tol:
                            is_new = False
                            break
                    if is_new:
                        solutions.append(float(sol))
            except:
                continue
        
        solutions.sort()
        all_solutions.extend(solutions)

    return sorted(list(set(round(sol/tol)*tol for sol in all_solutions)))

In [52]:
import numpy as np
import os
from PIL import Image, ImageDraw, ImageFont
import imageio
from sympy import symbols, nsolve, cos, sin, sqrt, pi
import matplotlib.pyplot as plt

# 系统参数
y11 = [2]
y22 = [2]
a = 0.5
zhi_1 = ['T']
E_ranges = [(150, 370)]
n3 = 0
n4 = 0
m = 0.5
h1 = 1

# 生成N值集合
N_set = np.linspace(1, 100, 100).astype(int)

# 主输出目录
main_output_dir = r"C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2"

# 创建基于参数的文件夹名称
def create_folder_name(y11, y22, N_set, E_ranges):
    y1_str = f"y1_{'_'.join(map(str, y11))}"
    y2_str = f"y2_{'_'.join(map(str, y22))}"
    N_str = f"N_{N_set[0]}-{N_set[-1]}"
    E_str = f"E_{E_ranges[0][0]}-{E_ranges[0][1]}"
    return f"T&R_{y1_str}_{y2_str}_{N_str}_{E_str}"

# 创建基于参数的动态图名称
def create_gif_name(y11, y22, N_set, E_ranges):
    y1_str = f"y1_{'_'.join(map(str, y11))}"
    y2_str = f"y2_{'_'.join(map(str, y22))}"
    N_str = f"N_{N_set[0]}-{N_set[-1]}"
    E_str = f"E_{E_ranges[0][0]}-{E_ranges[0][1]}"
    return f"T&R_Animation_{y1_str}_{y2_str}_{N_str}_{E_str}.gif"

# 创建文件夹
folder_name = create_folder_name(y11, y22, N_set, E_ranges)
output_dir = os.path.join(main_output_dir, folder_name)
os.makedirs(output_dir, exist_ok=True)

# 1. 生成所有静态图像
for N in N_set:
    # 计算能量级别
    energy_levels = calculate_energy_levels(N, y11[0], y22[0], a, E_ranges[0][0], E_ranges[0][1])
    
    # 计算传输系数
    E_values = np.linspace(E_ranges[0][0], E_ranges[0][1], 1000)
    T_values = [Transmission_coefficient_of_N_modified(E, y11[0], y22[0], N, n3, n4, a) for E in E_values]

    # 创建图形
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # 绘制传输系数曲线
    ax.plot(E_values, T_values, label='Transmission (T)', color='blue')
    
    # 添加能量级别线
    for x in energy_levels:
        ax.axvline(x, color='g', linestyle='-', linewidth=0.5, alpha=0.3)

    # 设置图形属性
    ax.set_xlabel('Energy (E)')
    ax.set_ylabel('Transmission Coefficient')
    ax.set_title(f'Transmission Coefficient (N={N}, y1={y11[0]}, y2={y22[0]})')
    ax.legend()
   # ax.grid(True)
    ax.set_xlim(E_ranges[0][0], E_ranges[0][1])
    ax.set_ylim(0, 20)
    ax.plot(E_values, T_values, 
        label='Transmission (T)', 
        color='#1f77b4',  # 使用更专业的蓝色
        linewidth=1.5,    # 适当加粗线条
        linestyle='-',    # 实线
        marker='',        # 无标记点
        alpha=0.8)        # 轻微透明

    # 保存图片
    img_name = f"T&R_N_{N}_y1_{y11[0]}_y2_{y22[0]}_E_{E_ranges[0][0]}-{E_ranges[0][1]}.png"
    img_path = os.path.join(output_dir, img_name)
    plt.savefig(img_path, dpi=300)
    plt.close()
    print(f"已保存: {img_path}")

# 2. 创建动态图
gif_name = create_gif_name(y11, y22, N_set, E_ranges)
gif_path = os.path.join(main_output_dir, gif_name)

# 获取所有图片文件并按N值排序
image_files = [f for f in os.listdir(output_dir) if f.endswith(".png")]
image_files.sort(key=lambda x: int(x.split("N_")[1].split("_")[0]))

# 准备字体
try:
    font = ImageFont.truetype("arial.ttf", 30)
except:
    font = ImageFont.load_default()

# 处理每张图片并创建帧列表
frames = []
for img_file in image_files:
    img_path = os.path.join(output_dir, img_file)
    
    with Image.open(img_path) as img:
        if img.mode in ('RGBA', 'LA'):
            background = Image.new('RGB', img.size, (255, 255, 255))
            background.paste(img, mask=img.split()[-1])
            img = background
        
        draw = ImageDraw.Draw(img)
        N_value = img_file.split("N_")[1].split("_")[0]
        text = f"N = {N_value}"
        
        # 计算文本位置
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        margin = 10
        x = img.width - text_width - margin
        y = margin
        
        # 绘制背景框和文本
        draw.rectangle([x-margin, y-margin, x+text_width+margin, y+text_height+margin], fill="black")
        draw.text((x, y), text, fill="white", font=font)
        
        frames.append(img)

# 保存为GIF
imageio.mimsave(gif_path, frames, format='GIF', duration=1)
print(f"动态图已保存至: {gif_path}")

已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_1_y1_2_y2_2_E_150-370.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_2_y1_2_y2_2_E_150-370.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_3_y1_2_y2_2_E_150-370.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_4_y1_2_y2_2_E_150-370.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_5_y1_2_y2_2_E_150-370.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_6_y1_2_y2_2_E_150-370.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_7_y1_2_y2_2_E_150-370.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_8_y1_2_y2_2_E_150-370.png
已保存: C:\Users\taoji\Desktop\结果\T&R实数与虚数_N细节_2\T&R_y1_2_y2_2_N_1-100_E_150-370\T&R_N_9_y1_2_y2_2_E_150-370.png
已保存: C:\Us